# CSCI 4253 / 5253 - Lab #4 - Patent Problem with Spark RDD - SOLUTION
<div>
 <h2> CSCI 4283 / 5253 
  <IMG SRC="https://www.colorado.edu/cs/profiles/express/themes/cuspirit/logo.png" WIDTH=50 ALIGN="right"/> </h2>
</div>

This [Spark cheatsheet](https://s3.amazonaws.com/assets.datacamp.com/blog_assets/PySpark_SQL_Cheat_Sheet_Python.pdf) is useful

In [1]:
from pyspark import SparkContext, SparkConf
import numpy as np
import operator

In [2]:
conf=SparkConf().setAppName("Lab4-rdd").setMaster("local[*]")
sc = SparkContext(conf=conf)

Using PySpark and RDD's on the https://coding.csel.io machines is slow -- most of the code is executed in Python and this is much less efficient than the java-based code using the PySpark dataframes. Be patient and trying using `.cache()` to cache the output of joins. You may want to start with a reduced set of data before running the full task. You can use the `sample()` method to extract just a sample of the data or use 

These two RDD's are called "rawCitations" and "rawPatents" because you probably want to process them futher (e.g. convert them to integer types, etc). 

The `textFile` function returns data in strings. This should work fine for this lab.

Other methods you use might return data in type `Byte`. If you haven't used Python `Byte` types before, google it. You can convert a value of `x` type byte into e.g. a UTF8 string using `x.decode('uft-8')`. Alternatively, you can use the `open` method of the gzip library to read in all the lines as UTF-8 strings like this:
```
import gzip
with gzip.open('cite75_99.txt.gz', 'rt',encoding='utf-8') as f:
    rddCitations = sc.parallelize( f.readlines() )
```
This is less efficient than using `textFile` because `textFile` would use the underlying HDFS or other file system to read the file across all the worker nodes while the using `gzip.open()...readlines()` will read all the data in the frontend and then distribute it to all the worker nodes.

In [3]:
rddCitations = sc.textFile("cite75_99.txt.gz")
rddPatents = sc.textFile("apat63_99.txt.gz")

The data looks like the following.

In [4]:
rddCitations.take(5)

['"CITING","CITED"',
 '3858241,956203',
 '3858241,1324234',
 '3858241,3398406',
 '3858241,3557384']

In [5]:
rddPatents.take(5)

['"PATENT","GYEAR","GDATE","APPYEAR","COUNTRY","POSTATE","ASSIGNEE","ASSCODE","CLAIMS","NCLASS","CAT","SUBCAT","CMADE","CRECEIVE","RATIOCIT","GENERAL","ORIGINAL","FWDAPLAG","BCKGTLAG","SELFCTUB","SELFCTLB","SECDUPBD","SECDLWBD"',
 '3070801,1963,1096,,"BE","",,1,,269,6,69,,1,,0,,,,,,,',
 '3070802,1963,1096,,"US","TX",,1,,2,6,63,,0,,,,,,,,,',
 '3070803,1963,1096,,"US","IL",,1,,2,6,63,,9,,0.3704,,,,,,,',
 '3070804,1963,1096,,"US","OH",,1,,2,6,63,,3,,0.6667,,,,,,,']

In [6]:
# rddPatents = rddPatents.sample(False, 0.05)
# rddCitations = rddCitations.sample(False, 0.05)

In other words, they are a single string with multiple CSV's. You will need to convert these to (K,V) pairs, probably convert the keys to `int` and so on. You'll need to `filter` out the header string as well since there's no easy way to extract all the lines except the first.

Create key value pairs (citing, state)

In [7]:
rddPatents = rddPatents.filter(
    lambda line: not line.startswith('"PATENT"')
)

In [8]:
patent_states = rddPatents.map(
    lambda line: line.split(",")
).map(
    lambda x: (
        int(x[0]),
        x[5].strip('"') if x[5].strip('"') != "" else None
    )
)

In [9]:
patent_states.take(5)

[(3070801, None),
 (3070802, 'TX'),
 (3070803, 'IL'),
 (3070804, 'OH'),
 (3070805, 'CA')]

Same with citations (citings, cited)

In [19]:

citation_pairs = rddCitations.filter(
    lambda line: not line.replace('"', '').startswith("CITING")
).map(
    lambda line: line.replace('"', '').split(",")
).map(
    lambda x: (int(x[0]), int(x[1]))
)

citation_pairs.take(5)

[(3858241, 956203),
 (3858241, 1324234),
 (3858241, 3398406),
 (3858241, 3557384),
 (3858241, 3634889)]

Joins on citing creating (citing (cited, citing_state))

In [20]:
citing_with_state = citation_pairs.join(patent_states)
citing_with_state.take(5)

[(3858560, (957631, 'IN')),
 (3859724, (2696848, 'PA')),
 (3859724, (2768428, 'PA')),
 (3859724, (3163926, 'PA')),
 (3859724, (3258039, 'PA'))]

swaps citing and cited
(citing, cited)

In [21]:
cited_lookup = citation_pairs.map(
    lambda x: (x[1], x[0])
)
cited_lookup.take(5)

[(956203, 3858241),
 (1324234, 3858241),
 (3398406, 3858241),
 (3557384, 3858241),
 (3634889, 3858241)]

Get cited with state: (cited,(citing, state))  

In [22]:
cited_with_state = cited_lookup.join(patent_states)

cited_data = cited_with_state.map(
    lambda x: (x[1][0], x[1][1])
)

Join the two RDDS   
citing with state -> (citing (cited) state)  
cited_data -> (citing, cited state)

In [ ]:
combined = citing_with_state.join(cited_data)
combined.take(5)

#citing cited citing state cited state

Count equal states

In [ ]:
same_state = combined.map(
    lambda x: (
        x[0],
        1 if x[1][0][1] is not None
                 and x[1][1] is not None
                 and x[1][0][1] == x[1][1]
        else 0
    )
)

In [ ]:
same_state_counts = same_state.reduceByKey(
    lambda a, b: a + b
)

In [ ]:
same_state_counts = same_state_counts.sortBy(
    lambda x: x[1],
    ascending=False
)

same_state_counts.take(10)

[(5979143, 8),
 (5322477, 7),
 (5871443, 7),
 (5921954, 6),
 (5751261, 5),
 (5960977, 5),
 (5457907, 5),
 (5637468, 4),
 (5773080, 4),
 (5777824, 4)]

format rdd to (Patent, original row)

In [ ]:
header = rddPatents.first()

patent_data = rddPatents.filter(lambda x: x != header)

patent_keyed = patent_data.map(
    lambda x: (x.split(",")[0], x)
)

In [ ]:
#make sure the counts have the same key
same_state_counts = same_state_counts.map(
    lambda x: (str(x[0]), x[1])
)

Join the counts back to the main table

In [ ]:
joined = patent_keyed.leftOuterJoin(same_state_counts)

result = joined.map(
    lambda x: x[1][0] + "," + str(x[1][1] if x[1][1] is not None else 0)
)
result_sorted = result.sortBy(
    lambda x: int(x[1].split(",")[-1]),
    ascending=False
)
result_sorted.take(10)

['3900005,1975,5709,1974,"GB","",17830,3,3,119,6,61,1,4,1,0,0,17.25,3,1,1,0.3333,0.25,0',
 '3900009,1975,5709,1974,"US","CA",,1,9,119,6,61,1,0,1,,0,,2,,,,,0',
 '3900079,1975,5709,1973,"FR","",61590,3,4,180,5,55,5,0,1,,0.32,,7,0,0,,,0',
 '3900089,1975,5709,1974,"US","MI",71655,2,8,192,5,53,4,6,0.25,0.5,0,10.3333,13.25,0,0,0,0,0',
 '3900103,1975,5709,1974,"GB","",266985,2,1,206,6,68,8,6,0,0.6111,,16.1667,21.25,0,0,0,0,0',
 '3900133,1975,5709,1974,"US","NJ",335445,2,7,221,5,51,5,1,0.6,0,0.6667,13,10.8,0,0,0,0,0',
 '3900343,1975,5709,1973,"NO","",531850,3,8,429,4,45,3,0,0.6667,,0.5,,18.3333,0,0,,,0',
 '3900385,1975,5709,1973,"JP","",379685,3,5,205,1,19,0,6,,0.7222,,13.1667,,,,0,0,0',
 '3900461,1975,5709,1972,"US","OR",597315,6,1,536,1,14,1,2,1,0.5,0,6,12,0,0,0,0,0',
 '3900464,1975,5709,1972,"CH","",254550,2,4,540,1,14,1,0,1,,0,,2,0,0,,,0']